# ingest_jdbc\nGenerated from 01_ingestion/ingest_jdbc.py for Databricks notebook import.\n

In [ ]:
"""Production-oriented generic JDBC ingestion adapter."""\n\nfrom __future__ import annotations\n\nimport logging\nimport os\nfrom dataclasses import dataclass\nfrom typing import Any, Mapping\n\nfrom pyspark.sql import DataFrame, SparkSession\n\nlogger = logging.getLogger(__name__)\n\n_REQUIRED_JDBC_PROFILE_KEYS = {"url", "driver", "fetchsize"}\n_PARTITION_OPTION_KEYS = {"numPartitions", "partitionColumn", "lowerBound", "upperBound"}\n_ALLOWED_EXTRA_OPTION_KEYS = {\n    "query",\n    "prepareQuery",\n    "queryTimeout",\n    "sessionInitStatement",\n    "pushDownPredicate",\n    "pushDownAggregate",\n    "pushDownLimit",\n    "pushDownOffset",\n    "pushDownTableSample",\n    "truncate",\n    "customSchema",\n    "isolationLevel",\n    "batchsize",\n    "fetchsize",\n    "numPartitions",\n    "partitionColumn",\n    "lowerBound",\n    "upperBound",\n    "queryTimeout",\n    "sessionInitStatement",\n    "query",\n    "prepareQuery",\n}\n\n\nclass JdbcIngestionConfigError(ValueError):\n    """Raised when JDBC ingestion configuration is invalid."""\n\n\nclass JdbcIngestionExecutionError(RuntimeError):\n    """Raised when JDBC ingestion execution fails."""\n\n\ndef _require_non_empty(value: Any, field_name: str) -> str:\n    if value is None:\n        raise JdbcIngestionConfigError(f"{field_name} is required")\n    text = str(value).strip()\n    if not text:\n        raise JdbcIngestionConfigError(f"{field_name} is required")\n    return text\n\n\ndef _to_spark_option(value: Any) -> str:\n    if isinstance(value, bool):\n        return str(value).lower()\n    if isinstance(value, (str, int, float)):\n        return str(value)\n    raise JdbcIngestionConfigError(\n        f"Unsupported Spark option value type for JDBC option: {type(value).__name__}"\n    )\n\n\ndef _validate_unknown_extra_options(extra_options: Mapping[str, Any] | None) -> None:\n    if not extra_options:\n        return\n\n    unknown = sorted(set(extra_options) - _ALLOWED_EXTRA_OPTION_KEYS)\n    if unknown:\n        raise JdbcIngestionConfigError(\n            f"Unsupported JDBC extra_options keys: {unknown}. "\n            f"Allowed keys are: {sorted(_ALLOWED_EXTRA_OPTION_KEYS)}"\n        )\n\n\ndef _resolve_env_value(env_var_name: str | None, field_name: str) -> str | None:\n    if not env_var_name:\n        return None\n    env_name = _require_non_empty(env_var_name, field_name)\n    value = os.getenv(env_name, "").strip()\n    if not value:\n        raise JdbcIngestionConfigError(\n            f"Environment variable '{env_name}' is required for {field_name}"\n        )\n    return value\n\n\n@dataclass(frozen=True)\nclass JdbcProfile:\n    url: str\n    driver: str\n    fetchsize: int\n    user_env: str | None = None\n    password_env: str | None = None\n\n    @classmethod\n    def from_dict(cls, jdbc_profile: Mapping[str, Any]) -> "JdbcProfile":\n        missing = [key for key in sorted(_REQUIRED_JDBC_PROFILE_KEYS) if not jdbc_profile.get(key)]\n        if missing:\n            raise JdbcIngestionConfigError(f"Missing JDBC profile fields: {missing}")\n\n        fetchsize = int(jdbc_profile["fetchsize"])\n        if fetchsize <= 0:\n            raise JdbcIngestionConfigError("jdbc_profile.fetchsize must be > 0")\n\n        return cls(\n            url=_require_non_empty(jdbc_profile.get("url"), "jdbc_profile.url"),\n            driver=_require_non_empty(jdbc_profile.get("driver"), "jdbc_profile.driver"),\n            fetchsize=fetchsize,\n            user_env=str(jdbc_profile.get("user_env")).strip() if jdbc_profile.get("user_env") else None,\n            password_env=str(jdbc_profile.get("password_env")).strip() if jdbc_profile.get("password_env") else None,\n        )\n\n\n@dataclass(frozen=True)\nclass SourceConfig:\n    jdbc_table: str\n    source_system: str | None = None\n    source_entity: str | None = None\n\n    @classmethod\n    def from_dict(cls, source: Mapping[str, Any]) -> "SourceConfig":\n        return cls(\n            jdbc_table=_require_non_empty(source.get("jdbc_table"), "source.jdbc_table"),\n            source_system=str(source.get("source_system")).strip() if source.get("source_system") else None,\n            source_entity=str(source.get("source_entity")).strip() if source.get("source_entity") else None,\n        )\n\n    @property\n    def source_id(self) -> str:\n        system = self.source_system or "unknown_system"\n        entity = self.source_entity or "unknown_entity"\n        return f"{system}.{entity}"\n\n\n@dataclass(frozen=True)\nclass ParallelReadOptions:\n    enabled: bool\n    options: dict[str, str]\n\n    @classmethod\n    def from_extra_options(cls, extra_options: Mapping[str, Any] | None) -> "ParallelReadOptions":\n        opts = dict(extra_options or {})\n        present = _PARTITION_OPTION_KEYS.intersection(opts)\n\n        if not present:\n            return cls(enabled=False, options={})\n\n        if present != _PARTITION_OPTION_KEYS:\n            missing = sorted(_PARTITION_OPTION_KEYS - present)\n            raise JdbcIngestionConfigError(\n                "Parallel JDBC read requires all partition options together. "\n                f"Missing keys: {missing}"\n            )\n\n        num_partitions = int(opts["numPartitions"])\n        if num_partitions <= 0:\n            raise JdbcIngestionConfigError("extra_options.numPartitions must be > 0")\n\n        partition_column = _require_non_empty(\n            opts["partitionColumn"], "extra_options.partitionColumn"\n        )\n\n        lower_bound = opts["lowerBound"]\n        upper_bound = opts["upperBound"]\n\n        return cls(\n            enabled=True,\n            options={\n                "numPartitions": _to_spark_option(num_partitions),\n                "partitionColumn": partition_column,\n                "lowerBound": _to_spark_option(lower_bound),\n                "upperBound": _to_spark_option(upper_bound),\n            },\n        )\n\n\n@dataclass(frozen=True)\nclass ResolvedJdbcConfig:\n    profile: JdbcProfile\n    source: SourceConfig\n    options: dict[str, str]\n\n\ndef build_jdbc_options(\n    jdbc_profile: Mapping[str, Any],\n    source: Mapping[str, Any],\n    extra_options: Mapping[str, Any] | None = None,\n) -> dict[str, str]:\n    _validate_unknown_extra_options(extra_options)\n\n    profile = JdbcProfile.from_dict(jdbc_profile)\n    source_cfg = SourceConfig.from_dict(source)\n    parallel = ParallelReadOptions.from_extra_options(extra_options)\n\n    options: dict[str, str] = {\n        "url": profile.url,\n        "dbtable": source_cfg.jdbc_table,\n        "driver": profile.driver,\n        "fetchsize": _to_spark_option(profile.fetchsize),\n    }\n\n    user = _resolve_env_value(profile.user_env, "jdbc_profile.user_env")\n    password = _resolve_env_value(profile.password_env, "jdbc_profile.password_env")\n\n    if user is not None:\n        options["user"] = user\n    if password is not None:\n        options["password"] = password\n\n    if parallel.enabled:\n        options.update(parallel.options)\n\n    for key, value in (extra_options or {}).items():\n        if key in _PARTITION_OPTION_KEYS:\n            continue\n        if value is None:\n            continue\n        options[key] = _to_spark_option(value)\n\n    return options\n\n\ndef resolve_jdbc_config(\n    jdbc_profile: Mapping[str, Any],\n    source: Mapping[str, Any],\n    extra_options: Mapping[str, Any] | None = None,\n) -> ResolvedJdbcConfig:\n    profile = JdbcProfile.from_dict(jdbc_profile)\n    source_cfg = SourceConfig.from_dict(source)\n    options = build_jdbc_options(jdbc_profile=jdbc_profile, source=source, extra_options=extra_options)\n    return ResolvedJdbcConfig(profile=profile, source=source_cfg, options=options)\n\n\ndef ingest_jdbc_batch(\n    spark: SparkSession,\n    jdbc_profile: Mapping[str, Any],\n    source: Mapping[str, Any],\n    extra_options: Mapping[str, Any] | None = None,\n) -> DataFrame:\n    resolved = resolve_jdbc_config(\n        jdbc_profile=jdbc_profile,\n        source=source,\n        extra_options=extra_options,\n    )\n\n    logger.info(\n        "Starting JDBC ingestion source_id=%s dbtable=%s url=%s driver=%s",\n        resolved.source.source_id,\n        resolved.source.jdbc_table,\n        resolved.profile.url,\n        resolved.profile.driver,\n    )\n\n    try:\n        reader = spark.read.format("jdbc").options(**resolved.options)\n        df = reader.load()\n\n        logger.info(\n            "Built JDBC DataFrame source_id=%s dbtable=%s",\n            resolved.source.source_id,\n            resolved.source.jdbc_table,\n        )\n        return df\n\n    except Exception as exc:\n        raise JdbcIngestionExecutionError(\n            f"JDBC ingestion failed for source_id={resolved.source.source_id} "\n            f"dbtable={resolved.source.jdbc_table} url={resolved.profile.url}: {exc}"\n        ) from exc\n